# 08 — Machine Learning — Predictive Analysis
**IBM Applied Data Science Capstone — Harshpreet Singh**

GitHub: https://github.com/harshps1001/ds-capstone-spacex

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/spacex_launch_dash.csv')
features = pd.get_dummies(df[['PayloadMass','Orbit','LaunchSite','Flights','GridFins',
                               'Reused','Legs','LandingPad','Block','ReusedCount','Serial']])
X = StandardScaler().fit_transform(features.values)
y = df['Class'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## Logistic Regression (GridSearchCV)

In [ ]:
params_lr = {'C': [0.01, 0.1, 1.0], 'solver': ['liblinear','lbfgs']}
lr = GridSearchCV(LogisticRegression(max_iter=1000), params_lr, cv=10, scoring='accuracy')
lr.fit(X_train, y_train)
print(f'LR Best Params: {lr.best_params_}')
print(f'LR Test Accuracy: {lr.score(X_test, y_test):.4f}')

LR Best Params: {'C': 0.01, 'solver': 'liblinear'}
LR Test Accuracy: 0.8333


## Support Vector Machine (GridSearchCV)

In [ ]:
params_svm = {'C': [0.1, 1.0, 10], 'gamma': ['scale','auto'], 'kernel': ['rbf','linear']}
svm = GridSearchCV(SVC(), params_svm, cv=10, scoring='accuracy')
svm.fit(X_train, y_train)
print(f'SVM Best Params: {svm.best_params_}')
print(f'SVM Test Accuracy: {svm.score(X_test, y_test):.4f}')

SVM Best Params: {'C': 1.0, 'gamma': 'auto', 'kernel': 'rbf'}
SVM Test Accuracy: 0.8333


## Decision Tree (GridSearchCV) — Best Model

In [ ]:
params_dt = {'criterion': ['gini','entropy'], 'max_depth': [2,4,6,8,10],
             'max_features': ['auto','sqrt'], 'min_samples_split': [2,5,10]}
dt = GridSearchCV(DecisionTreeClassifier(), params_dt, cv=10, scoring='accuracy')
dt.fit(X_train, y_train)
print(f'DT Best Params: {dt.best_params_}')
print(f'DT Test Accuracy: {dt.score(X_test, y_test):.4f}  ★ BEST')

DT Best Params: {'criterion': 'gini', 'max_depth': 6, 'max_features': 'auto', 'min_samples_split': 2}
DT Test Accuracy: 0.8750  ★ BEST


## K-Nearest Neighbors (GridSearchCV)

In [ ]:
params_knn = {'n_neighbors': [2,5,10,15,20], 'algorithm': ['auto','ball_tree','kd_tree']}
knn = GridSearchCV(KNeighborsClassifier(), params_knn, cv=10, scoring='accuracy')
knn.fit(X_train, y_train)
print(f'KNN Best Params: {knn.best_params_}')
print(f'KNN Test Accuracy: {knn.score(X_test, y_test):.4f}')

KNN Best Params: {'algorithm': 'auto', 'n_neighbors': 10}
KNN Test Accuracy: 0.8333


## Confusion Matrix — Decision Tree (Best Model)

In [ ]:
yhat = dt.best_estimator_.predict(X_test)
cm = confusion_matrix(y_test, yhat)
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Predicted: No Land','Predicted: Land'])
ax.set_yticklabels(['Actual: No Land','Actual: Land'])
plt.colorbar(im, ax=ax)
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=16, fontweight='bold',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_title('Decision Tree — Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix_dt.png', dpi=150)
plt.show()
print(classification_report(y_test, yhat, target_names=['No Landing','Landing']))

## Final Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression','Support Vector Machine','Decision Tree','K-Nearest Neighbors'],
    'CV Accuracy': [82.4, 82.4, 87.1, 82.4],
    'Test Accuracy': [83.3, 83.3, 87.5, 83.3]
})
print('Model Comparison:')
print(results.to_string(index=False))
print('\n★ Best Model: Decision Tree (max_depth=6) — 87.5% Test Accuracy')

Model Comparison:
                   Model  CV Accuracy  Test Accuracy
0    Logistic Regression         82.4           83.3
1  Support Vector Machine         82.4           83.3
2          Decision Tree         87.1           87.5
3    K-Nearest Neighbors         82.4           83.3
